# Phase 2B — Automated lifecycle classification

This notebook applies a transparent, deterministic rule-based lifecycle classifier to the frozen 548-row text corpus. It does **not** run sentiment analysis, lifecycle findings, topic modelling, VADER, or LDA.

Controlled labels:

- `pre_adoption`: expectations, barriers, feasibility, purchase intent, infrastructure needs, or comparisons before adoption.
- `procurement`: tenders, bids, contracts, order awards, fleet purchase, delivery commitments, subsidy schemes, and institutional procurement.
- `post_use`: actual travel, booking, service, operations, breakdown, delay, refund, staff, comfort, route, or maintenance experience.
- `general_unspecified`: no sufficiently strong lifecycle evidence, weak evidence, or unresolved ties.

Evidence precedence is explicit experience > hypothetical language, while explicit procurement language remains procurement evidence. Speaker role and source context may support a classification but never determine it alone.


In [1]:
# Optional environment cell. The classifier itself uses only pandas, numpy, and the Python standard library.
INSTALL_PACKAGES = False
if INSTALL_PACKAGES:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', '../requirements.txt'])


In [2]:
from pathlib import Path
import hashlib, json, re, random
from collections import defaultdict
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'PHASE1_DECISIONS.md').exists() and (candidate / '01_data').exists():
            return candidate
    raise FileNotFoundError('Project root not found')

ROOT = find_root()
INPUT = ROOT / '01_data/processed/ev_bus_text_corpus_frozen.csv'
OUT_DATA = ROOT / '01_data/processed/ev_bus_text_corpus_lifecycle_v1.csv'
OUT_DIR = ROOT / '04_outputs/text/lifecycle'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

input_hash_before=sha256(INPUT)
df=pd.read_csv(INPUT)
assert len(df)==548 and df['comment_id'].is_unique
frozen_columns=list(df.columns)
print('Input:', INPUT.relative_to(ROOT), df.shape, input_hash_before)


## Rule dictionary and scoring logic

Rules use protected multiword phrases before individual keywords. Broad terms such as “battery”, “cost”, or “service” are weak context only and cannot independently override a stronger explicit stage. A stage normally needs a score of at least 2. A top-score tie is assigned `general_unspecified` unless explicit actual-use evidence clearly outranks hypothetical language. Multiple meaningful stages are retained in `lifecycle_secondary_flags` and sent to manual review.

Confidence:

- **high**: explicit evidence with a clear margin;
- **medium**: sufficient but less decisive evidence, or a clear absence of lifecycle evidence for `general_unspecified`;
- **low**: weak, close, tied, or unresolved evidence.

Observed Hinglish tokens are retained as contextual evidence only. Hinglish is never automatically excluded or assigned a lifecycle stage.


In [3]:
RULES = {
 'pre_adoption': [
  ('PRE01', r'\brange anxiety\b',3,'strong'), ('PRE02',r'\bcharging infrastructure\b',3,'strong'),
  ('PRE03',r'\b(charging availability|charging station|charging network)\b',3,'strong'),
  ('PRE04',r'\b(worth buying|should (?:i |we )?buy|planning to buy|purchase intent)\b',3,'strong'),
  ('PRE05',r'\b(need|needs|require|required|lack|without|not enough)\b.{0,35}\b(charging|infrastructure|range)\b',3,'strong'),
  ('PRE06',r'\b(diesel|cng)\b.{0,30}\b(compare|comparison|versus|vs\.?|better|worse|than)\b|\b(compare|comparison|versus|vs\.?|better|worse|than)\b.{0,30}\b(diesel|cng)\b',3,'strong'),
  ('PRE07',r'\b(affordability|feasibility|viability|adoption barrier|future of electric bus(?:es)?)\b',2,'medium'),
  ('PRE08',r'\b(battery swap|battery swapping|solar panel|solar penal)\b',2,'medium'),
  ('PRE09',r'\b(expensive|price|cost|range|battery|infrastructure|adoption|pollution|environment)\b',1,'weak'),
  ('PRE10',r'\b(chahiye|kaise|kya)\b.{0,35}\b(bus|battery|charging|electric)\b|\b(bus|battery|charging|electric)\b.{0,35}\b(chahiye|kaise|kya)\b',1,'weak')],
 'procurement': [
  ('PRO01',r'\b(tender|tenders|bid|bidding|lowest bidder|\bl1\b)\b',3,'strong'),
  ('PRO02',r'\b(gross cost contract|\bgcc\b|per km|per kilometre|assured kilometres?)\b',3,'strong'),
  ('PRO03',r'\b(tender awarded|contract awarded|order received|order awarded|won (?:the )?order|secured (?:an? |the )?order)\b',4,'strong'),
  ('PRO04',r'\b(government order|order allocation|fleet purchase|subsidy scheme|delivery commitment|order book)\b',3,'strong'),
  ('PRO05',r'\b(government|state|agency|corporation)\b.{0,35}\b(order|contract|tender|procure)\b',2,'medium'),
  ('PRO06',r'\b(supply|deliver|delivery)\b.{0,30}\b\d+\s+(?:electric\s+)?buses\b',2,'medium'),
  ('PRO07',r'\b(procure|procurement|contract|award|order|fleet)\b',1,'weak'),
  ('PRO08',r'\b(paisa|kharidne|theka)\b.{0,30}\b(bus|tender|order)\b|\b(bus|tender|order)\b.{0,30}\b(paisa|kharidne|theka)\b',1,'weak')],
 'post_use': [
  ('POST01',r'\b(i travelled|i traveled|i used|my experience|on my route|i took (?:the |a )?bus|travelled in|traveled in)\b',4,'strong'),
  ('POST02',r'\b(broke down|breakdown|not working|stopped working|customer care|service centre|service center)\b',3,'strong'),
  ('POST03',r'\b(refund|cancelled|canceled|cancellation|pnr|booking|ticket)\b',3,'strong'),
  ('POST04',r'\b(ride quality|seat comfort|ac was|air conditioning|luggage|driver|conductor|staff behaviour|staff behavior)\b',3,'strong'),
  ('POST05',r'\b(delay|delayed|waiting|waited|late|route|maintenance|charging time)\b',2,'medium'),
  ('POST06',r'\b(my|meri|maine|mai|i|we)\b.{0,45}\b(bus|ride|travel|trip|booking|ticket|route|refund|driver|staff|delay)\b',2,'medium'),
  ('POST07',r'\b(bus|ride|travel|trip|booking|ticket|route|refund|driver|staff|delay)\b.{0,45}\b(my|meri|maine|mai|i|we)\b',2,'medium'),
  ('POST08',r'\b(meri bus thi|jarha hu|ja raha hu|paisa wapas|nahi aayi|nahi aya)\b',3,'strong'),
  ('POST09',r'\b(service|comfort|smooth|clean|dirty|crowded|experience)\b',1,'weak')]
}

OBSERVED_HINGLISH = sorted({t.strip().lower() for x in df['hinglish_terms_detected'].fillna('') for t in str(x).split(';') if t.strip()})
dictionary = {
 'version':'lifecycle_rules_v1.0', 'seed':SEED,
 'controlled_labels':['pre_adoption','procurement','post_use','general_unspecified'],
 'minimum_stage_score':2,
 'tie_policy':'Assign general_unspecified for unresolved top-score ties; retain all tied/multiple stages as secondary flags and require manual review.',
 'evidence_precedence':'Explicit actual-use evidence outranks hypothetical language; explicit procurement phrases remain procurement evidence. Speaker/source support is never decisive alone.',
 'observed_hinglish_tokens':OBSERVED_HINGLISH,
 'rules':{stage:[{'rule_id':i,'pattern':p,'weight':w,'strength':s} for i,p,w,s in rules] for stage,rules in RULES.items()}
}
with open(OUT_DIR/'lifecycle_keyword_dictionary.json','w',encoding='utf-8') as f:
    json.dump(dictionary,f,indent=2,ensure_ascii=False)
print('Rules:', {k:len(v) for k,v in RULES.items()}, 'Observed Hinglish tokens:',len(OBSERVED_HINGLISH))


In [4]:
def classify(row):
    text=str(row.get('sentiment_text','') if pd.notna(row.get('sentiment_text','')) else row.get('raw_text',''))
    low=text.lower()
    scores=defaultdict(int); matches=defaultdict(list); rule_ids=defaultdict(list); strengths=defaultdict(list)
    for stage,rules in RULES.items():
        for rid,pat,weight,strength in rules:
            found=re.search(pat,low,flags=re.I)
            if found:
                scores[stage]+=weight; term=found.group(0).strip()
                matches[stage].append(term); rule_ids[stage].append(rid); strengths[stage].append(strength)
    # Context support: never creates a stage without text evidence.
    if scores['procurement']>=2 and str(row.get('speaker_role_final',''))=='media_or_research': scores['procurement']+=1
    if scores['post_use']>=2 and str(row.get('speaker_role_final',''))=='audience_user': scores['post_use']+=1
    vals={s:int(scores[s]) for s in ['pre_adoption','procurement','post_use']}
    top=max(vals.values()); tied=[s for s,v in vals.items() if v==top and top>=2]
    meaningful=[s for s,v in vals.items() if v>=2]
    explicit_post='strong' in strengths['post_use']
    explicit_proc='strong' in strengths['procurement']
    ambiguous=False
    if top<2:
        label='general_unspecified'
    elif len(tied)>1:
        # Only resolve when actual-use evidence is explicit and competing evidence is merely weak.
        if explicit_post and all('strong' not in strengths[s] for s in tied if s!='post_use'):
            label='post_use'
        elif explicit_proc and all('strong' not in strengths[s] for s in tied if s!='procurement'):
            label='procurement'
        else:
            label='general_unspecified'; ambiguous=True
    else:
        label=tied[0]
    ordered=sorted(vals,key=lambda s:(-vals[s],s)); margin=vals[ordered[0]]-vals[ordered[1]]
    if label=='general_unspecified':
        confidence='low' if ambiguous or top==1 else 'medium'
    elif top>=4 and margin>=2 and ('strong' in strengths[label]): confidence='high'
    elif top>=2 and margin>=1: confidence='medium'
    else: confidence='low'
    secondary=[s for s in meaningful if s!=label]
    hinglish=bool(str(row.get('hinglish_terms_detected','')).strip())
    unknown=str(row.get('speaker_role_final',''))=='unknown'
    underrep=str(row.get('entity_name_canonical','')) in ['EKA Mobility','Switch Mobility']
    manual=confidence=='low' or ambiguous or len(meaningful)>1 or (hinglish and top>=1) or unknown or underrep
    evidence=[]; used=[]
    for s in ['pre_adoption','procurement','post_use']:
        evidence += [f'{s}:{x}' for x in dict.fromkeys(matches[s])]
        used += rule_ids[s]
    return pd.Series({
      'pre_adoption_score':vals['pre_adoption'],'procurement_score':vals['procurement'],'post_use_score':vals['post_use'],
      'lifecycle_stage_auto':label,'lifecycle_confidence':confidence,
      'lifecycle_evidence_terms':'; '.join(evidence),'lifecycle_rule_ids':'; '.join(dict.fromkeys(used)),
      'lifecycle_secondary_flags':'; '.join(secondary),'lifecycle_ambiguous':ambiguous,
      'lifecycle_manual_review_required':manual})

classified=df.apply(classify,axis=1)
out=pd.concat([df.copy(),classified],axis=1)
out['lifecycle_consumer_sentiment_scope']=out['consumer_sentiment_eligible_final'].astype(bool)
out['lifecycle_consumer_topic_scope']=out['consumer_topic_eligible_final'].astype(bool)
out['lifecycle_industry_topic_scope']=out['industry_topic_eligible_final'].astype(bool)
out.to_csv(OUT_DATA,index=False)
print(out['lifecycle_stage_auto'].value_counts().to_string())
print(out['lifecycle_confidence'].value_counts().to_string())


## Distribution tables

Percentages are calculated within the displayed group and scope. `small_sample_flag` is `True` whenever the relevant group contains fewer than 15 rows; such cells must not be interpreted as stable subgroup findings.


In [5]:
LABEL_ORDER=['pre_adoption','procurement','post_use','general_unspecified']
def dist_table(data, group_cols):
    base=data.groupby(group_cols,dropna=False).size().rename('sample_size').reset_index()
    counts=data.groupby(group_cols+['lifecycle_stage_auto'],dropna=False).size().rename('row_count').reset_index()
    result=counts.merge(base,on=group_cols,how='left')
    result['percent']=100*result['row_count']/result['sample_size']
    result['small_sample_flag']=result['sample_size']<15
    return result

overall=out['lifecycle_stage_auto'].value_counts().reindex(LABEL_ORDER,fill_value=0).rename_axis('lifecycle_stage_auto').reset_index(name='row_count')
overall['sample_size']=len(out); overall['percent']=100*overall['row_count']/len(out); overall['small_sample_flag']=False
overall.to_csv(OUT_DIR/'lifecycle_distribution.csv',index=False)

scopes={'all_corpus':pd.Series(True,index=out.index),'consumer_sentiment':out['lifecycle_consumer_sentiment_scope'],
        'consumer_topic':out['lifecycle_consumer_topic_scope'],'industry_topic':out['lifecycle_industry_topic_scope']}
scope_tables=[]
for name,mask in scopes.items():
    x=out.loc[mask].copy(); x['scope_name']=name; scope_tables.append(dist_table(x,['scope_name']))
pd.concat(scope_tables,ignore_index=True).to_csv(OUT_DIR/'lifecycle_distribution_by_scope.csv',index=False)

dimension_tables=[]
for dim in ['entity_name_canonical','entity_type','oem_group','speaker_role_final','provenance']:
    for scope_name,mask in scopes.items():
        x=out.loc[mask].copy(); x['scope_name']=scope_name; x['dimension']=dim; x['group_value']=x[dim].fillna('Missing').astype(str)
        dimension_tables.append(dist_table(x,['scope_name','dimension','group_value']))
pd.concat(dimension_tables,ignore_index=True).to_csv(OUT_DIR/'lifecycle_distribution_by_entity.csv',index=False)

platform_tables=[]
for scope_name,mask in scopes.items():
    x=out.loc[mask].copy(); x['scope_name']=scope_name; x['platform_group']=x['platform_canonical'].fillna('Mixed/Unknown')
    platform_tables.append(dist_table(x,['scope_name','platform_group']))
pd.concat(platform_tables,ignore_index=True).to_csv(OUT_DIR/'lifecycle_distribution_by_platform.csv',index=False)
print('Distribution files saved')


## Deterministic manual-validation sample

The sample uses seed 42. It prioritises ambiguous/multistage cases, low confidence, observed Hinglish with lifecycle signals, EKA Mobility and Switch Mobility, and manual-review flags. It then fills gaps across lifecycle class, confidence, entity type, OEM group, platform, and consumer/industry scopes while capping `general_unspecified` where feasible. Human-review columns are intentionally blank.


In [6]:
rng=np.random.default_rng(SEED)
sampled=[]; reasons=defaultdict(set)
def add(indices,reason,cap=None):
    candidates=[i for i in indices if i not in sampled]
    candidates=sorted(candidates,key=lambda i:str(out.at[i,'comment_id']))
    if cap is not None and len(candidates)>cap:
        candidates=list(rng.choice(candidates,size=cap,replace=False)); candidates=sorted(candidates,key=lambda i:str(out.at[i,'comment_id']))
    for i in candidates:
        if len(sampled)>=120: break
        sampled.append(i); reasons[i].add(reason)

add(out.index[out['entity_name_canonical'].isin(['EKA Mobility','Switch Mobility'])],'underrepresented_oem')
add(out.index[out['lifecycle_ambiguous']],'ambiguous',30)
add(out.index[out['lifecycle_secondary_flags'].fillna('').ne('')],'multistage',30)
add(out.index[out['lifecycle_confidence'].eq('low')],'low_confidence',35)
add(out.index[out['hinglish_terms_detected'].fillna('').ne('') & out['lifecycle_evidence_terms'].fillna('').ne('')],'hinglish_signal',20)
add(out.index[out['lifecycle_manual_review_required']],'manual_review_required',35)

# Coverage minima by predicted class and confidence.
for col,values,target in [('lifecycle_stage_auto',LABEL_ORDER,18),('lifecycle_confidence',['high','medium','low'],12)]:
    for value in values:
        have=sum(out.at[i,col]==value for i in sampled)
        add(out.index[out[col].eq(value)],f'coverage_{col}_{value}',max(0,target-have))

# One row for every available entity type, OEM group, platform, and scope condition.
for col in ['entity_type','oem_group','platform_canonical']:
    for value in sorted(out[col].dropna().astype(str).unique()):
        if not any(str(out.at[i,col])==value for i in sampled): add(out.index[out[col].astype(str).eq(value)],f'coverage_{col}_{value}',1)
for col in ['lifecycle_consumer_sentiment_scope','lifecycle_consumer_topic_scope','lifecycle_industry_topic_scope']:
    for value in [True,False]:
        if (out[col]==value).any() and not any(bool(out.at[i,col])==value for i in sampled): add(out.index[out[col].eq(value)],f'coverage_{col}_{value}',1)

# Fill deterministically, preferring non-general rows to avoid general-class domination.
remaining=[i for i in out.index if i not in sampled]
remaining=sorted(remaining,key=lambda i:(out.at[i,'lifecycle_stage_auto']=='general_unspecified',str(out.at[i,'comment_id'])))
for i in remaining:
    if len(sampled)>=120: break
    sampled.append(i); reasons[i].add('seeded_stratified_fill')

sample=out.loc[sampled].copy()
sample['lifecycle_sampling_reason']=['; '.join(sorted(reasons[i])) for i in sampled]
keep=['comment_id','entity_name_canonical','entity_type','oem_group','platform_canonical','speaker_role_final','provenance','raw_text','sentiment_text',
      'pre_adoption_score','procurement_score','post_use_score','lifecycle_stage_auto','lifecycle_confidence','lifecycle_evidence_terms','lifecycle_rule_ids',
      'lifecycle_secondary_flags','lifecycle_ambiguous','lifecycle_manual_review_required','lifecycle_consumer_sentiment_scope','lifecycle_consumer_topic_scope',
      'lifecycle_industry_topic_scope','lifecycle_sampling_reason']
sample=sample[keep]
for c in ['human_lifecycle_stage','human_confidence','human_evidence_note','reviewer_name','review_date']:
    sample[c]=''
sample.to_csv(OUT_DIR/'lifecycle_manual_validation_sample.csv',index=False)
print('Manual sample:',len(sample)); print(sample['lifecycle_stage_auto'].value_counts().to_string())


## Examples for auditability

The following display is descriptive only. It shows rule evidence for each automated class plus ambiguous/low-confidence cases; it does not constitute findings.


In [7]:
examples=pd.concat([out[out.lifecycle_stage_auto.eq(x)].head(2) for x in LABEL_ORDER])
display(examples[['comment_id','sentiment_text','lifecycle_stage_auto','lifecycle_confidence','lifecycle_evidence_terms','lifecycle_rule_ids']])
display(out[(out.lifecycle_ambiguous)|(out.lifecycle_confidence.eq('low'))][['comment_id','sentiment_text','lifecycle_stage_auto','lifecycle_confidence','lifecycle_secondary_flags']].head(10))


In [8]:
# Build the classification report from persisted classifications.
stage_counts=out['lifecycle_stage_auto'].value_counts().reindex(LABEL_ORDER,fill_value=0)
conf_counts=out['lifecycle_confidence'].value_counts().reindex(['high','medium','low'],fill_value=0)
rule_counts=out['lifecycle_rule_ids'].fillna('').str.split('; ').explode(); rule_counts=rule_counts[rule_counts.ne('')].value_counts()
secondary_n=out['lifecycle_secondary_flags'].fillna('').ne('').sum()
ambiguous_n=int(out['lifecycle_ambiguous'].sum())
sample_class=sample['lifecycle_stage_auto'].value_counts().to_dict(); sample_conf=sample['lifecycle_confidence'].value_counts().to_dict()

lines=['# Lifecycle Classification Report','',
 '## Scope and provenance','',
 f'- Input: `01_data/processed/ev_bus_text_corpus_frozen.csv`',
 f'- Input SHA-256: `{input_hash_before}`',
 f'- Rows classified: **{len(out)}**; unique `comment_id`: **{out.comment_id.nunique()}**.',
 '- All 548 frozen rows were classified. No row was deleted, and the frozen input was not modified.',
 '- This output is automated classification only. It is **not** a validated lifecycle finding.', '',
 '## Dictionary and scoring','',
 f'- Dictionary version: `lifecycle_rules_v1.0`; rules: pre-adoption {len(RULES["pre_adoption"])}, procurement {len(RULES["procurement"])}, post-use {len(RULES["post_use"])}.',
 '- Rules match protected phrases and contextual expressions in `sentiment_text` (falling back to `raw_text` only if needed). `topic_model_text` is not used.',
 '- Weights are additive and non-negative. A stage normally requires score >= 2. Weak isolated terms do not force a stage.',
 '- Explicit actual-use evidence outranks hypothetical language. Explicit tender/order/contract language remains procurement evidence. Speaker/source support can add one point only after text evidence exists.',
 '- Unresolved top-score ties become `general_unspecified`; tied and multiple meaningful stages are retained for manual review.',
 '- Confidence is high for explicit evidence with a clear margin, medium for sufficient less-decisive evidence or clear absence of lifecycle evidence, and low for weak/close/unresolved cases.','',
 '## Automated distribution (all 548 rows)','']
for k,v in stage_counts.items(): lines.append(f'- `{k}`: {int(v)} ({100*v/len(out):.1f}%)')
lines += ['', '## Confidence and review flags','']
for k,v in conf_counts.items(): lines.append(f'- `{k}`: {int(v)} ({100*v/len(out):.1f}%)')
lines += [f'- Ambiguous/tied cases: {ambiguous_n}.',f'- Multiple-stage secondary flags: {int(secondary_n)}.',f'- Automated manual-review-required flags: {int(out.lifecycle_manual_review_required.sum())}.','',
 '## Rule firing counts','']
for k,v in rule_counts.items(): lines.append(f'- `{k}`: {int(v)} rows')
lines += ['', '## Manual-validation sample','',
 f'- Sample size: **{len(sample)}**, deterministic seed **{SEED}**, unique IDs: **{sample.comment_id.nunique()}**.',
 f'- Class composition: `{json.dumps(sample_class,sort_keys=True)}`.',
 f'- Confidence composition: `{json.dumps(sample_conf,sort_keys=True)}`.',
 '- Sampling prioritises ambiguous, multistage, low-confidence, Hinglish-signal, EKA Mobility, Switch Mobility, and other manual-review cases, then fills coverage across classes, confidence, entity type, OEM group, platform, and analysis scopes.',
 '- Human-review fields are blank by design.','',
 '## Illustrative automated examples','']
for label in LABEL_ORDER:
    z=out[out.lifecycle_stage_auto.eq(label)].head(2)
    for _,r in z.iterrows():
        txt=str(r.sentiment_text).replace('\n',' ')[:180]
        lines.append(f'- `{label}` / `{r.comment_id}`: {txt} — evidence: `{r.lifecycle_evidence_terms or "none"}`.')
lines += ['', '## Limitations and required human validation','',
 '- Keyword and regex rules cannot reliably resolve sarcasm, implicit meaning, negation scope, quoted material, or comments that discuss several lifecycle stages.',
 '- Indian English and Hinglish vary in spelling and grammar. Only tokens observed in this frozen corpus are documented; code-switching itself never determines a label.',
 '- Generic words such as cost, battery, service, route, order, and experience are context-sensitive. The conservative threshold reduces false specificity but increases `general_unspecified` assignments.',
 '- Platform, entity, OEM-group, and provenance imbalance can distort apparent distributions. Every subgroup table identifies sample sizes and flags groups below 15 rows.',
 '- Speaker role and source type are imperfect metadata and are supporting context only.',
 '- The manual-validation sample must be reviewed before any lifecycle result is used in the MBA report. Automated outputs must not be reported as validated findings.',
 '- No sentiment, VADER, LDA, topic modelling, or synthetic-data analysis was performed.']
(OUT_DIR/'lifecycle_classification_report.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('Report saved')


In [9]:
# Final validation
valid_labels=set(LABEL_ORDER); valid_conf={'high','medium','low'}
assert len(out)==548 and out['comment_id'].is_unique
assert list(out.columns[:len(frozen_columns)])==frozen_columns
pd.testing.assert_series_equal(out['raw_text'],df['raw_text'],check_names=False)
for c in frozen_columns: pd.testing.assert_series_equal(out[c],df[c],check_names=False)
assert set(out['lifecycle_stage_auto'])<=valid_labels and set(out['lifecycle_confidence'])<=valid_conf
assert (out[['pre_adoption_score','procurement_score','post_use_score']]>=0).all().all()
non_general=out['lifecycle_stage_auto'].ne('general_unspecified')
assert out.loc[non_general,'lifecycle_evidence_terms'].fillna('').ne('').all()
assert out.loc[non_general,'lifecycle_rule_ids'].fillna('').ne('').all()
assert (out['lifecycle_consumer_sentiment_scope']==out['consumer_sentiment_eligible_final'].astype(bool)).all()
assert (out['lifecycle_consumer_topic_scope']==out['consumer_topic_eligible_final'].astype(bool)).all()
assert (out['lifecycle_industry_topic_scope']==out['industry_topic_eligible_final'].astype(bool)).all()
assert len(sample)==120 and sample['comment_id'].is_unique
for c in ['human_lifecycle_stage','human_confidence','human_evidence_note','reviewer_name','review_date']:
    assert sample[c].fillna('').eq('').all()
assert sha256(INPUT)==input_hash_before
required=[OUT_DATA,*[OUT_DIR/x for x in ['lifecycle_distribution.csv','lifecycle_distribution_by_scope.csv','lifecycle_distribution_by_entity.csv','lifecycle_distribution_by_platform.csv','lifecycle_keyword_dictionary.json','lifecycle_manual_validation_sample.csv','lifecycle_classification_report.md']]]
assert all(p.exists() for p in required)
print('FINAL VALIDATION PASSED')
print(f'Retained rows: {len(out)} | unique IDs: {out.comment_id.nunique()}')
print('Lifecycle classes:', stage_counts.to_dict())
print('Confidence:', conf_counts.to_dict())
print(f'Ambiguous: {ambiguous_n} | multistage: {secondary_n} | manual sample: {len(sample)}')
print('Frozen input SHA-256 unchanged:', sha256(INPUT))
